# 02 — How did the agent reach 2022-03-09?

We audit one real Luna review of synthetic case `SYNX03`.

- **Task:** find the earliest diagnosis date.
- **Agent answer:** `20220309`.
- **Synthetic gold:** `20220309`.
- **First conclusion:** the final answer is correct.

A correct answer can still come from a bad process. We therefore ask: **what did the
agent look at, what did it judge, and which step is still risky?**

Think of the raw trace as a long receipt. It proves what happened. The Decision Chain
groups that receipt into a short set of questions a human can judge.


In [ ]:
from pathlib import Path
import json
import os
import re

from IPython.display import Markdown, display
from acr.mvp.human_review import human_review_view
from acr.mvp.ledger import SemanticaLedger

START = Path.cwd().resolve()
ROOT = START if (START / "pyproject.toml").is_file() else START.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

candidates = [
    ROOT / "runs/notebook-live-20260827/ledger.json",
    ROOT / "runs/postdoc-study/ledger.json",
    ROOT / "runs/policy-experiment-20260827/experiment-ledger.json",
]
LEDGER_PATH = Path(os.environ.get(
    "ACR_AUDIT_LEDGER", next((str(path) for path in candidates if path.is_file()), "")
))
assert LEDGER_PATH.is_file(), "Run Notebook 1 first or set ACR_AUDIT_LEDGER"
ledger = SemanticaLedger(LEDGER_PATH)

preferred = (
    "20260827T135252029492Z_SYNX03_STORE_390_date_of_initial_diagnosis_policy_bundle"
)
run_id = os.environ.get("ACR_AUDIT_RUN_ID")
if run_id is None and ledger.selected_analysis(preferred):
    run_id = preferred
if run_id is None:
    selections = ledger.graph.find_nodes(node_type="AnalysisSelection")
    selected_runs = [
        str((row.get("metadata") or {}).get("run_id")) for row in selections
        if "SYNX03" in str((row.get("metadata") or {}).get("run_id"))
    ]
    assert selected_runs, "Select a reconstructed SYNX03 run first"
    run_id = selected_runs[-1]
analysis_id = ledger.selected_analysis(run_id)
assert analysis_id, "This walkthrough requires one explicitly selected reconstruction"

run_dir = LEDGER_PATH.parent / run_id
artifact = ledger.load_analysis_artifact(run_id, analysis_id)
view = human_review_view(ledger, run_id, analysis_id, run_dir=run_dir)
events = [json.loads(line) for line in (run_dir / "trace.jsonl").read_text().splitlines()]
protocol_path = run_dir / "layer2_codex.jsonl"
protocol_count = sum(1 for line in protocol_path.read_text().splitlines() if line.strip())
result = json.loads((run_dir / "result.json").read_text())
episodes = artifact["episodes"]
cycles = artifact["cycles"]

def one_line(value, limit=105):
    text = " ".join(str(value or "").split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def table(rows, columns):
    def clean(value):
        return one_line(value, 145).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(clean(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

def tool_events(name):
    return [row for row in events if row.get("tool") == name]


## 1. What does the raw trace look like?

The full trace has 22 observable events. Reading every JSON field is possible, but it
is not how a reviewer should start. First group the receipt by purpose.


In [ ]:
inventory_pages = [len((row.get("result") or {}).get("documents") or [])
                   for row in tool_events("list_documents")]
queries = [str((row.get("args") or {}).get("query")) for row in tool_events("search")]
hit_notes = {
    str(hit.get("note_id"))
    for row in tool_events("search")
    for hit in (row.get("result") or {}).get("hits") or []
}
process = [
    {"stage": "Inventory", "what happened": (
        f"Two pages: {' + '.join(map(str, inventory_pages))} = "
        f"{sum(inventory_pages)} note headers"
    )},
    {"stage": "Search", "what happened": f"Keywords: {', '.join(queries)}"},
    {"stage": "Candidates", "what happened": f"{len(hit_notes)} unique notes surfaced"},
    {"stage": "Read", "what happened": f"Opened {len(tool_events('read'))} notes"},
    {"stage": "Judge", "what happened": (
        f"Recorded {len(tool_events('record_finding'))} evidence judgments"
    )},
    {"stage": "Proof", "what happened": (
        f"Saved {len(tool_events('record_evidence'))} exact evidence spans"
    )},
    {"stage": "Submit", "what happened": "FOUND → 20220309"},
]
display(Markdown(table(process, [("stage", "Stage"), ("what happened", "Raw trace says") ])))

raw = tool_events("record_finding")[1]
args = raw["args"]
compact_event = {
    "seq": raw["seq"],
    "kind": raw["kind"],
    "tool": raw["tool"],
    "args": {
        "note_id": args.get("note_id"),
        "standing": args.get("standing"),
        "because": args.get("because"),
    },
    "result": {
        "ok": raw.get("ok"),
        "testimony_ref": (raw.get("result") or {}).get("testimony_ref"),
        "server_sealed_receipt": bool(
            (raw.get("result") or {}).get("decision_receipt")
        ),
    },
}
display(Markdown("### One actual event (trimmed only for display)"))
display(Markdown(
    "```json\n" + json.dumps(compact_event, ensure_ascii=False, indent=2) + "\n```"
))


A raw event is excellent evidence: it tells us which note, which tool, which choice,
and whether the server sealed the record. But it does not tell the reviewer where one
important judgment ends and the next begins.

## 2. Turn the receipt into eight questions

Luna reconstructs fixed ReAct cycles into **Decision Episodes**. One episode means one
consequential question that one human can mark right, wrong, or uncertain.

```text
312-note inventory
        ↓
4 keyword searches
        ↓
3 candidate notes
        ↓
2/14 ❌ suspicious   3/9 ✅ biopsy   3/11 ✅ physician diagnosis
        ↓
earliest qualifying date = 3/9
        ↓
submit 20220309
```


In [ ]:
def human_question(ep):
    subject = ep.get("decision_subject")
    text = (ep.get("material_question") or "").lower()
    if subject == "retrieval_inventory":
        return "Was the chart inventory complete?"
    if subject == "retrieval_query_batch":
        return "Which keywords should we use to search for diagnosis evidence?"
    if subject == "retrieval_document_set":
        return "Which surfaced notes should be opened?"
    if subject == "evidence_item" and "suspicious" in text:
        return "Can the 2/14 suspicious cytology establish diagnosis?"
    if subject == "evidence_item" and "pathology report" in text:
        return "Can the 3/9 definitive biopsy establish diagnosis?"
    if subject == "evidence_item" and "physician" in text:
        return "Can the 3/11 physician diagnosis establish diagnosis?"
    if subject == "evidence_relationship":
        return "Which of the three dates is the earliest qualifying date?"
    if subject == "case_sufficiency":
        return "Is there enough evidence to stop and submit?"
    return one_line(ep.get("material_question"), 80)

def human_choice(ep):
    subject = ep.get("decision_subject")
    decision = str(ep.get("decision") or "")
    if subject == "retrieval_inventory":
        return f"Inventory all {sum(inventory_pages)} note headers"
    if subject == "retrieval_query_batch":
        return "Search diagnosis, cancer, malignancy, carcinoma"
    if subject == "retrieval_document_set":
        return f"Open the {len(tool_events('read'))} surfaced candidate notes"
    if decision == "merely_mentions":
        return "No — suspicious only"
    if decision == "can_establish":
        return "Yes — qualifying evidence"
    if subject == "evidence_relationship":
        return "Choose 2022-03-09"
    if subject == "case_sufficiency":
        return "Stop and submit 20220309"
    return one_line(decision, 72)

def provenance_label(ep):
    provenance = ep.get("field_provenance") or {}
    if all(provenance.get(field) == "SELF_REPORTED"
           for field in ("material_question", "decision", "decision_rationale")):
        return "Agent said this at runtime"
    if provenance.get("decision") == "DETERMINISTIC_DERIVED_FROM_EXECUTION":
        return "Action observed; reason reconstructed by Luna"
    return "Luna reconstructed"

verdicts = {
    "retrieval_inventory": "✓ Complete after page 2",
    "retrieval_query_batch": "⚠ Main audit point: are four terms enough?",
    "retrieval_document_set": "⚠ Notes opened, but selection reason was not recorded",
    "evidence_relationship": "✓ Applies the earliest-qualifying rule correctly",
    "case_sufficiency": "△ Correct, but inherits the search-coverage risk",
}
evidence_verdicts = [
    "✓ Correctly rejects ambiguous cytology alone",
    "✓ Definitive pathology qualifies",
    "✓ Physician diagnosis qualifies, but is later",
]
evidence_index = 0
decision_rows = []
for index, ep in enumerate(episodes, 1):
    subject = str(ep.get("decision_subject"))
    verdict = verdicts.get(subject, "Review")
    if subject == "evidence_item":
        verdict = evidence_verdicts[evidence_index]
        evidence_index += 1
    decision_rows.append({
        "n": index,
        "question": human_question(ep),
        "choice": human_choice(ep),
        "audit": f"{verdict} · {provenance_label(ep)}",
    })

display(Markdown(table(decision_rows, [
    ("n", "#"), ("question", "Question for the reviewer"),
    ("choice", "Agent's choice"), ("audit", "Human audit reading + source"),
])))


## 3. Follow the clinical decision

The three evidence judgments are the heart of the answer:

1. `2022-02-14`: atypical cells were only **suspicious**; biopsy was recommended. This
   note alone does not establish the date.
2. `2022-03-09`: the biopsy's final diagnosis is squamous cell carcinoma. This does
   establish the date.
3. `2022-03-11`: the physician says the mass clinically represents malignancy. This
   also qualifies, but it is later.
4. The task asks for the **earliest qualifying** date, so `2022-03-09` wins.

This is the same explanation a careful human reviewer would ask a colleague to give.


## 4. Where should the human spend time?

Not on the final date comparison. The weak point is earlier: **the agent chose four
search terms.** Those calls really happened, and they found the three decisive notes.
But the policy says what evidence counts; it does not prove that these four terms cover
every possible clinical wording.


In [ ]:
search_episode = next(
    ep for ep in episodes if ep.get("decision_subject") == "retrieval_query_batch"
)
search_event_ids = set(search_episode.get("source_event_ids") or [])
search_events = [
    row for row in events if f"layer1:{row.get('seq')}" in search_event_ids
    and row.get("tool") == "search"
]
display(Markdown(
    f"- **Agent's question:** {search_episode['material_question']}\n"
    f"- **Agent's choice:** {search_episode['decision']}\n"
    f"- **Agent's stated reason:** {search_episode['decision_rationale']}\n"
    f"- **Raw proof:** events {', '.join(str(row['seq']) for row in search_events)} "
    f"executed `{', '.join(queries)}` and surfaced {len(hit_notes)} unique notes.\n"
    "- **Reviewer question:** would a different wording escape all four searches?"
))
display(Markdown(
    "**If the answer is yes—or we cannot rule it out—improve the retrieval guideline "
    "here.** Changing the "
    "later date-conflict rule would not fix a missed note."
))


## 5. Is the Decision level better to read?

Yes—for choosing **where to audit**. No—as a replacement for execution evidence.


In [ ]:
annotations = artifact["cycle_annotations"]
if isinstance(annotations, list):
    annotations = {row["cycle_id"]: row for row in annotations}
assigned = [cycle_id for ep in episodes for cycle_id in ep["source_cycle_ids"]]
assigned += list(artifact.get("mechanical_cycle_ids") or [])
assert len(assigned) == len(cycles) == len(set(assigned))
traceable = sum(bool(ep.get("source_event_ids")) for ep in episodes)
runtime_explained = sum(
    (ep.get("field_provenance") or {}).get("decision") == "SELF_REPORTED"
    for ep in episodes
)
reconstructed_only = len(episodes) - runtime_explained

display(Markdown(
    f"```text\n{protocol_count} Codex protocol records (harness detail)\n"
    f"→ {len(events)} observable Langtrace events\n"
    f"→ {len(cycles)} fixed ReAct cycles\n"
    f"→ {len(episodes)} human-auditable decisions\n```\n\n"
    f"- {runtime_explained}/{len(episodes)} choices were explicitly stated by the "
    f"agent at runtime.\n"
    f"- {reconstructed_only}/{len(episodes)} choice was recovered from observed "
    "actions; its explanation is visibly labeled as Luna reconstruction.\n"
    f"- {traceable}/{len(episodes)} decisions link back to raw events.\n"
    f"- {len(cycles)}/{len(cycles)} cycles are accounted for exactly once."
))
display(Markdown(
    "**Bottom line:** use the Decision Chain to find the questionable step. Then use "
    "the raw trace and provenance to verify what actually happened."
))
